# 🚗 ANPR Pipeline — License Plate Detection + OCR

**Self-contained Kaggle Notebook** — no file uploads needed.

## Setup checklist
1. **Add your Datasets** (top-right “Add data” button in Kaggle):
   - Your **model dataset** containing `plate_detector_yolo11n.pt`
   - Your **video dataset** containing `video.mp4`
2. **Enable GPU**: *Session options → Accelerator → GPU P100 or T4*
3. **Run All** cells top-to-bottom.

---
### How to upload your models and video to Kaggle Datasets
1. Go to [kaggle.com/datasets/new](https://www.kaggle.com/datasets/new)
2. Upload your files (drag-and-drop). You can use one combined dataset.
3. Publish (can be **Private**).
4. In this notebook → click **Add data** → search your dataset → attach it.
5. Kaggle mounts it at `/kaggle/input/<dataset-slug>/`
6. Update `MODEL_PATH` and `VIDEO_PATH` in Cell 3 if auto-discovery fails.

In [ ]:
# Cell 1: Install dependencies
%pip install -q ultralytics "fast-plate-ocr" opencv-python-headless scipy
print('Dependencies installed.')


In [ ]:
# Cell 2: Environment check
import os, platform, torch
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
else:
    print('No GPU — go to Session options -> Accelerator -> GPU for faster inference.')
os.chdir('/kaggle/working')
print('Working dir:', os.getcwd())

In [ ]:
# Cell 3: Configuration  (UPDATE PATHS IF NEEDED)
import glob

def _find_file(pattern):
    matches = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    return matches[0] if matches else None

MODEL_PATH = _find_file('plate_detector_yolo11n.pt') or _find_file('*.pt')
VIDEO_PATH = _find_file('video.mp4') or _find_file('*.mp4')

# Override here if auto-discovery picks the wrong file:
# MODEL_PATH = '/kaggle/input/YOUR-DATASET-SLUG/plate_detector_yolo11n.pt'
# VIDEO_PATH = '/kaggle/input/YOUR-VIDEO-SLUG/video.mp4'

print('MODEL_PATH :', MODEL_PATH)
print('VIDEO_PATH :', VIDEO_PATH)

if not MODEL_PATH or not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError('Model (.pt) not found. Attach your model dataset and/or update MODEL_PATH.')
if not VIDEO_PATH or not os.path.isfile(VIDEO_PATH):
    raise FileNotFoundError('Video (.mp4) not found. Attach your video dataset and/or update VIDEO_PATH.')

OUTPUT_VIDEO       = '/kaggle/working/output_annotated.mp4'
EVENT_CSV_LOG      = '/kaggle/working/plate_log.csv'
RAW_READS_CSV      = '/kaggle/working/plate_raw_reads.csv'
SUMMARY_CSV_LOG    = '/kaggle/working/plate_summary.csv'
CROPPED_PLATES_DIR = '/kaggle/working/cropped_plates'

OCR_MODEL_NAME         = 'cct-s-v2-global-model'
CONFIDENCE             = 0.25
SKIP_FRAMES            = 1
OCR_CONF_THRESHOLD     = 0.40
EARLY_STOP_CONF        = 0.90
OCR_INTERVAL_ACTIVE    = 1
OCR_INTERVAL_CONFIRMED = 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MIN_PLATE_CHARS = 4
MAX_PLATE_CHARS = 10
UPSCALE_TARGET_HEIGHT = 80
IMAGE_INFERENCE_SIZE  = 640
TRACK_IOU_THRESHOLD   = 0.3
TRACK_MAX_AGE         = 30
TRACK_DRAW_MAX_MISSED = 5
RIGHT_ROI_FRACTION    = 0.25
RIGHT_ROI_UPSCALE     = 1.8
RIGHT_ROI_CONF_SCALE  = 0.65
MERGE_IOU_THRESH      = 0.35
DISTANT_HEIGHT_PX     = 32
OCR_ANGLE_RETRY_DEGREES = (8, 15)
print(f'Config ready | Device: {DEVICE}')

In [ ]:
# Cell 4: Helper functions
import cv2, re, csv, time
import numpy as np
from datetime import datetime
from typing import List, Tuple, Optional, Dict
from scipy.optimize import linear_sum_assignment

def upscale_if_small(crop, target_height):
    h, w = crop.shape[:2]
    if h == 0 or w == 0: return crop
    if h < target_height:
        scale = target_height / float(h)
        new_w = max(1, int(w * scale))
        crop = cv2.resize(crop, (new_w, target_height),
                          interpolation=cv2.INTER_LANCZOS4 if scale > 1 else cv2.INTER_AREA)
    return crop

def apply_clahe(gray, clip_limit=3.0):
    return cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8,8)).apply(gray)

def unsharp_mask(img, amount=1.5, sigma=1.0):
    return cv2.addWeighted(img, 1+amount, cv2.GaussianBlur(img,(0,0),sigma), -amount, 0)

def denoise(img):
    return cv2.fastNlMeansDenoisingColored(img,None,7,7,7,21) if len(img.shape)==3 \
           else cv2.fastNlMeansDenoising(img,None,7,7,21)

def gamma_correct(img, gamma=1.4):
    table = np.array([((i/255.0)**(1.0/gamma))*255 for i in range(256)],dtype='uint8')
    return cv2.LUT(img, table)

def is_low_quality(gray):
    return cv2.Laplacian(gray,cv2.CV_64F).var()<60.0, gray.std()<40.0

def generate_enhanced_variants(crop_bgr, is_distant=False):
    variants=[]; target_h=UPSCALE_TARGET_HEIGHT*(2 if is_distant else 1)
    crop_up=upscale_if_small(crop_bgr,target_h); variants.append(crop_up)
    sh,sw=crop_bgr.shape[:2]
    if sh>0 and sw>0 and sh<target_h:
        hrh=int(target_h*1.5); hrw=max(1,int(sw*hrh/sh))
        variants.append(cv2.resize(crop_bgr,(hrw,hrh),interpolation=cv2.INTER_CUBIC))
    gray=cv2.cvtColor(crop_up,cv2.COLOR_BGR2GRAY)
    blurry,low_c=is_low_quality(gray)
    clahe_sharp=unsharp_mask(apply_clahe(gray),amount=1.2,sigma=1.0)
    variants.append(cv2.cvtColor(clahe_sharp,cv2.COLOR_GRAY2BGR))
    if low_c or blurry or is_distant: variants.append(gamma_correct(denoise(crop_up),gamma=1.5))
    if blurry or is_distant: variants.append(unsharp_mask(crop_up,amount=2.0,sigma=1.5))
    if is_distant:
        bil=cv2.bilateralFilter(crop_up,d=7,sigmaColor=50,sigmaSpace=50)
        sc=apply_clahe(cv2.cvtColor(bil,cv2.COLOR_BGR2GRAY),clip_limit=4.5)
        variants.append(cv2.cvtColor(sc,cv2.COLOR_GRAY2BGR))
    return variants

def rotate_bound(image, angle):
    h,w=image.shape[:2]; cx,cy=w/2.0,h/2.0
    M=cv2.getRotationMatrix2D((cx,cy),angle,1.0)
    cos,sin=abs(M[0,0]),abs(M[0,1])
    nw,nh=int(h*sin+w*cos),int(h*cos+w*sin)
    M[0,2]+=nw/2.0-cx; M[1,2]+=nh/2.0-cy
    return cv2.warpAffine(image,M,(nw,nh),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_REPLICATE)

def generate_angle_variants(crop_bgr, is_distant=False):
    th=UPSCALE_TARGET_HEIGHT*(2 if is_distant else 1)
    cup=upscale_if_small(crop_bgr,th)
    return [rotate_bound(cup,a*d) for a in OCR_ANGLE_RETRY_DEGREES for d in (-1,1)]

def iou(a,b):
    xa,ya=max(a[0],b[0]),max(a[1],b[1]); xb,yb=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0.0,xb-xa)*max(0.0,yb-ya)
    aa=max(0.0,a[2]-a[0])*max(0.0,a[3]-a[1]); ab=max(0.0,b[2]-b[0])*max(0.0,b[3]-b[1])
    d=aa+ab-inter; return inter/d if d>0 else 0.0

_OCR_ERROR_PRINTED=False

def clean_plate_text(raw):
    if not raw: return None
    t=re.sub(r'[^A-Za-z0-9]','',raw.upper().replace('_',''))
    return t if MIN_PLATE_CHARS<=len(t)<=MAX_PLATE_CHARS else None

def safe_filename_fragment(text):
    return re.sub(r'[^A-Za-z0-9_-]','',text) or 'UNKNOWN'

def run_ocr_engine(recognizer, image_bgr):
    global _OCR_ERROR_PRINTED
    if image_bgr is None or not hasattr(image_bgr, 'shape') or image_bgr.shape[0] < 10 or image_bgr.shape[1] < 10:
        return []
    def _predict(arr):
        results = []
        for pred in recognizer.run(arr, return_confidence=True):
            pt = getattr(pred, 'plate', None)
            cp = getattr(pred, 'char_probs', None)
            if pt:
                conf = float(np.mean(cp)) if cp is not None and len(cp) > 0 else 0.0
                results.append((pt, conf))
        return results
    try:
        if len(image_bgr.shape) == 2:
            rgb = cv2.cvtColor(image_bgr, cv2.COLOR_GRAY2RGB)
        elif image_bgr.shape[2] == 4:
            rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGRA2RGB)
        else:
            rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        return _predict(rgb)
    except Exception as e:
        if not _OCR_ERROR_PRINTED:
            _OCR_ERROR_PRINTED = True
            print(f'[OCR ERROR] {type(e).__name__}: {e}')
        return []

def recognize_variants(recognizer,variants,raw_writer,frame_idx,track_id):
    best_text,best_score=None,0.0
    for v in variants:
        if v is None or not hasattr(v, 'shape') or v.shape[0] < 10 or v.shape[1] < 10:
            continue
        for raw,conf in run_ocr_engine(recognizer,v):
            cleaned=clean_plate_text(raw)
            accepted=bool(cleaned and conf>=OCR_CONF_THRESHOLD)
            raw_writer.writerow([datetime.now().isoformat(timespec='seconds'),
                frame_idx,track_id,raw,cleaned or '',f'{conf:.3f}',accepted])
            if cleaned and conf>best_score: best_text,best_score=cleaned,conf
    return best_text,best_score

print('Helper functions ready.')


In [ ]:
# Cell 5: Track + IoUTracker
class Track:
    _next_id=1
    def __init__(self,bbox,frame_idx):
        self.id=Track._next_id; Track._next_id+=1
        self.bbox=bbox; self.time_since_update=0; self.hits=1
        self.first_frame=frame_idx; self.last_frame=frame_idx; self.last_ocr_frame=-999
        self.text_conf_sum={}; self.text_count={}
        self.best_text=None; self.best_score=0.0
        self.last_raw_text=None; self.last_raw_score=0.0
        self.best_crop=None; self.best_crop_score=0.0; self.crop_saved_for_text=None
        self.kf=cv2.KalmanFilter(8,4)
        self.kf.transitionMatrix=np.eye(8,dtype=np.float32)
        for i in range(4): self.kf.transitionMatrix[i,i+4]=1.0
        self.kf.measurementMatrix=np.eye(4,8,dtype=np.float32)
        self.kf.processNoiseCov=np.eye(8,dtype=np.float32)*1e-2
        self.kf.measurementNoiseCov=np.eye(4,dtype=np.float32)*1e-1
        cx,cy,w,h=self._to_cxcywh(bbox)
        self.kf.statePost=np.array([cx,cy,w,h,0,0,0,0],dtype=np.float32).reshape(8,1)
    @staticmethod
    def _to_cxcywh(b): return (b[0]+b[2])/2,(b[1]+b[3])/2,b[2]-b[0],b[3]-b[1]
    @staticmethod
    def _to_xyxy(cx,cy,w,h): return cx-w/2,cy-h/2,cx+w/2,cy+h/2
    def predict(self):
        p=self.kf.predict()
        self.bbox=self._to_xyxy(float(p[0]),float(p[1]),max(float(p[2]),1.),max(float(p[3]),1.))
        self.time_since_update+=1; return self.bbox
    def update(self,bbox,frame_idx):
        cx,cy,w,h=self._to_cxcywh(bbox)
        self.kf.correct(np.array([[cx],[cy],[w],[h]],dtype=np.float32))
        self.bbox=bbox; self.time_since_update=0; self.hits+=1; self.last_frame=frame_idx
    def register_ocr(self,text,score):
        prev=self.best_text
        self.text_conf_sum[text]=self.text_conf_sum.get(text,0.)+score
        self.text_count[text]=self.text_count.get(text,0)+1
        best=max(self.text_conf_sum,key=lambda t:self.text_conf_sum[t])
        self.best_text=best; self.best_score=self.text_conf_sum[best]/self.text_count[best]
        return self.best_text!=prev
    def maybe_update_crop(self,crop,score):
        if score>self.best_crop_score: self.best_crop_score=score; self.best_crop=crop.copy()

class IoUTracker:
    def __init__(self,iou_threshold=TRACK_IOU_THRESHOLD,max_age=TRACK_MAX_AGE):
        self.tracks=[]; self.all_tracks=[]
        self.iou_threshold=iou_threshold; self.max_age=max_age
    def update(self,detections,frame_idx):
        for t in self.tracks: t.predict()
        if not self.tracks:
            for d in detections: self.tracks.append(Track(d,frame_idx))
            return self.tracks
        if not detections:
            active=[]
            for t in self.tracks:
                (active if t.time_since_update<=self.max_age else self.all_tracks).append(t)
            self.tracks=active; return self.tracks
        cost=np.zeros((len(self.tracks),len(detections)),dtype=np.float32)
        for i,t in enumerate(self.tracks):
            for j,d in enumerate(detections):
                iou_v=iou(t.bbox,d)
                ct=((t.bbox[0]+t.bbox[2])/2,(t.bbox[1]+t.bbox[3])/2)
                cd=((d[0]+d[2])/2,(d[1]+d[3])/2)
                dist=np.sqrt((ct[0]-cd[0])**2+(ct[1]-cd[1])**2)
                dt=np.sqrt((t.bbox[2]-t.bbox[0])**2+(t.bbox[3]-t.bbox[1])**2)
                dd=np.sqrt((d[2]-d[0])**2+(d[3]-d[1])**2)
                cost[i,j]=(1.-iou_v)+0.4*(dist/max((dt+dd)/2.,1e-5))
        ri,ci=linear_sum_assignment(cost)
        mt,md=set(),set()
        for r,c in zip(ri,ci):
            if cost[r,c]<=2.0: self.tracks[r].update(detections[c],frame_idx); mt.add(r); md.add(c)
        for j,d in enumerate(detections):
            if j not in md: self.tracks.append(Track(d,frame_idx))
        active=[]
        for t in self.tracks:
            (active if t.time_since_update<=self.max_age else self.all_tracks).append(t)
        self.tracks=active; return self.tracks

print('Track + IoUTracker ready.')

In [ ]:
# Cell 6: Detection function + crop saver
def detect_plates(model,frame,width,height):
    dets=[]
    res=model(frame,conf=CONFIDENCE,device=DEVICE,imgsz=IMAGE_INFERENCE_SIZE,half=(DEVICE=='cuda'),verbose=False)
    if res[0].boxes is not None and len(res[0].boxes)>0:
        for (x1,y1,x2,y2),c in zip(res[0].boxes.xyxy.cpu().numpy(),res[0].boxes.conf.cpu().numpy()):
            dets.append([float(x1),float(y1),float(x2),float(y2),float(c),'main'])
    rx1=int(width*(1-RIGHT_ROI_FRACTION)); roi=frame[0:height,rx1:width]
    if roi.size>0:
        rh,rw=roi.shape[:2]
        rup=cv2.resize(roi,(int(rw*RIGHT_ROI_UPSCALE),int(rh*RIGHT_ROI_UPSCALE)),interpolation=cv2.INTER_CUBIC)
        rr=model(rup,conf=CONFIDENCE*RIGHT_ROI_CONF_SCALE,device=DEVICE,imgsz=IMAGE_INFERENCE_SIZE,half=(DEVICE=='cuda'),verbose=False)
        if rr[0].boxes is not None and len(rr[0].boxes)>0:
            for (x1,y1,x2,y2),c in zip(rr[0].boxes.xyxy.cpu().numpy(),rr[0].boxes.conf.cpu().numpy()):
                fx1=x1/RIGHT_ROI_UPSCALE+rx1; fy1=y1/RIGHT_ROI_UPSCALE
                fx2=x2/RIGHT_ROI_UPSCALE+rx1; fy2=y2/RIGHT_ROI_UPSCALE
                if not any(d[5]=='main' and iou((fx1,fy1,fx2,fy2),d[:4])>MERGE_IOU_THRESH for d in dets):
                    dets.append([fx1,fy1,fx2,fy2,float(c),'right_boost'])
    return dets

def save_track_crop(track,directory):
    if track.best_text is None or track.best_crop is None: return
    if track.crop_saved_for_text==track.best_text: return
    fname=f'ID{track.id}_{safe_filename_fragment(track.best_text)}.jpg'
    try:
        cv2.imwrite(os.path.join(directory,fname),track.best_crop)
        track.crop_saved_for_text=track.best_text
    except Exception as e: print(f'[WARNING] crop save failed: {e}')

print('detect_plates + save_track_crop ready.')

In [ ]:
# Cell 7: Load models
from ultralytics import YOLO
from fast_plate_ocr import LicensePlateRecognizer
print(f'Loading YOLO: {MODEL_PATH}')
yolo_model=YOLO(MODEL_PATH)
print('YOLO loaded.')
print(f'Loading OCR: {OCR_MODEL_NAME}')
ocr=LicensePlateRecognizer(OCR_MODEL_NAME)
print('OCR loaded.')

In [ ]:
# Cell 8: Run ANPR pipeline
Track._next_id=1
_OCR_ERROR_PRINTED=False
os.makedirs(CROPPED_PLATES_DIR,exist_ok=True)

cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise RuntimeError(f'Cannot open: {VIDEO_PATH}')

fps=int(cap.get(cv2.CAP_PROP_FPS)) or 25
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {width}x{height} @ {fps}fps | {total_frames} frames')

fourcc=cv2.VideoWriter_fourcc(*'mp4v')
out_writer=cv2.VideoWriter(OUTPUT_VIDEO,fourcc,fps,(width,height))
if not out_writer.isOpened(): print('[WARNING] VideoWriter failed.'); out_writer=None

tracker=IoUTracker(); right_zone_x=int(width*(1-RIGHT_ROI_FRACTION))
frame_count=processed=0; start_time=time.time()

with open(EVENT_CSV_LOG,'w',newline='',encoding='utf-8') as ef, \
     open(RAW_READS_CSV,'w',newline='',encoding='utf-8') as rf:
    ew=csv.writer(ef); ew.writerow(['timestamp','frame','track_id','plate_text','confidence','zone'])
    rw=csv.writer(rf); rw.writerow(['timestamp','frame','track_id','raw_text','cleaned_text','confidence','accepted'])
    try:
        while True:
            ret,frame=cap.read()
            if not ret: break
            frame_count+=1
            if SKIP_FRAMES>0 and frame_count%(SKIP_FRAMES+1)!=0:
                if out_writer: out_writer.write(frame)
                continue
            processed+=1
            dets=detect_plates(yolo_model,frame,width,height)
            active=tracker.update([(d[0],d[1],d[2],d[3]) for d in dets],frame_count)
            ann=frame.copy()
            for t in active:
                if t.time_since_update>TRACK_DRAW_MAX_MISSED: continue
                x1,y1,x2,y2=[int(max(0,v)) for v in t.bbox]
                x2,y2=min(width,x2),min(height,y2)
                if x2<=x1 or y2<=y1: continue
                is_dist=(x1>=right_zone_x) or ((y2-y1)<DISTANT_HEIGHT_PX)
                do_ocr=(t.time_since_update==0 and
                    ((t.best_score<EARLY_STOP_CONF and frame_count-t.last_ocr_frame>=OCR_INTERVAL_ACTIVE) or
                     (t.best_score>=EARLY_STOP_CONF and frame_count-t.last_ocr_frame>=OCR_INTERVAL_CONFIRMED)))
                if do_ocr:
                    crop=frame[y1:y2,x1:x2]
                    if crop is not None and crop.size > 0 and crop.shape[0] >= 10 and crop.shape[1] >= 10:
                        t.last_ocr_frame=frame_count
                        bt,bs=recognize_variants(ocr,generate_enhanced_variants(crop,is_dist),rw,frame_count,t.id)
                        if bs<OCR_CONF_THRESHOLD:
                            at,as_=recognize_variants(ocr,generate_angle_variants(crop,is_dist),rw,frame_count,t.id)
                            if as_>bs: bt,bs=at,as_
                        rf.flush()
                        if bt:
                            t.last_raw_text=bt; t.last_raw_score=bs; t.maybe_update_crop(crop,bs)
                        if bt and bs>=OCR_CONF_THRESHOLD:
                            changed=t.register_ocr(bt,bs)
                            if changed:
                                ew.writerow([datetime.now().isoformat(timespec='seconds'),
                                    frame_count,t.id,t.best_text,f'{t.best_score:.3f}',
                                    'right_boost' if is_dist else 'main'])
                                ef.flush()
                            save_track_crop(t,CROPPED_PLATES_DIR)
                has_c=t.best_text is not None; has_p=not has_c and t.last_raw_text is not None
                color=(0,255,0) if has_c else (0,165,255)
                label=t.best_text if has_c else (f'{t.last_raw_text}?' if has_p else None)
                cv2.rectangle(ann,(x1,y1),(x2,y2),color,2)
                if label:
                    (tw,th),_=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.8,2)
                    tx=x1+max(0,((x2-x1)-tw)//2); ty=y1+max(th,((y2-y1)+th)//2)
                    cv2.putText(ann,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,0),4,cv2.LINE_AA)
                    cv2.putText(ann,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,0.8,color,2,cv2.LINE_AA)
                else:
                    cv2.putText(ann,f'ID {t.id}',(x1,max(15,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,255),1,cv2.LINE_AA)
            cv2.line(ann,(right_zone_x,0),(right_zone_x,height),(255,200,0),1)
            cv2.putText(ann,f'Processed:{processed} | Tracks:{len(active)}',(10,30),cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,255,0),2)
            if out_writer: out_writer.write(ann)
            if processed%100==0:
                el=time.time()-start_time; spd=processed/el if el>0 else 0
                print(f'[{frame_count}/{total_frames}] Tracks:{len(active)} | {spd:.1f}fps')
    finally:
        cap.release()
        if out_writer: out_writer.release()
        ft=tracker.all_tracks+tracker.tracks; vt=[t for t in ft if t.best_text]
        for t in vt: save_track_crop(t,CROPPED_PLATES_DIR)
        with open(SUMMARY_CSV_LOG,'w',newline='',encoding='utf-8') as f:
            w=csv.writer(f)
            w.writerow(['track_id','best_plate_text','confidence','first_frame','last_frame','cropped_image'])
            for t in sorted(vt,key=lambda t:t.id):
                img=f'ID{t.id}_{safe_filename_fragment(t.best_text)}.jpg' if t.best_crop else ''
                w.writerow([t.id,t.best_text,f'{t.best_score:.3f}',t.first_frame,t.last_frame,img])
        el=time.time()-start_time
        print('='*55)
        print('ANPR Pipeline Complete')
        print(f'  Processed: {processed} frames | {len(vt)} unique plates')
        print(f'  Time: {el:.1f}s ({processed/max(el,1):.1f} fps avg)')
        if DEVICE=='cuda': print(f'  VRAM: {torch.cuda.memory_allocated()/1024**2:.1f} MB')
        print('='*55)


In [ ]:
# Cell 9: Preview results
import pandas as pd
from PIL import Image
from IPython.display import display

if os.path.isfile(SUMMARY_CSV_LOG):
    df=pd.read_csv(SUMMARY_CSV_LOG)
    print(f'Plate Summary ({len(df)} unique plates):')
    display(df)

if os.path.isfile(OUTPUT_VIDEO):
    cap2=cv2.VideoCapture(OUTPUT_VIDEO)
    fc=int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
    if cap2.isOpened() and fc>0:
        cap2.set(cv2.CAP_PROP_POS_FRAMES,fc//2)
        ok,pf=cap2.read()
        if ok:
            print('Sample annotated frame (mid-video):')
            display(Image.fromarray(cv2.cvtColor(pf,cv2.COLOR_BGR2RGB)))
    cap2.release()

crops=sorted(glob.glob(os.path.join(CROPPED_PLATES_DIR,'*.jpg')))[:6]
if crops:
    print(f'Sample cropped plates ({len(crops)} shown):')
    for p in crops: print(os.path.basename(p)); display(Image.open(p))

In [ ]:
# Cell 10: Package all outputs into ZIP
# All files in /kaggle/working are downloadable from the Output panel.
# This cell also bundles them for convenience.
import zipfile
ZIP_PATH='/kaggle/working/anpr_results.zip'
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [OUTPUT_VIDEO,EVENT_CSV_LOG,RAW_READS_CSV,SUMMARY_CSV_LOG]:
        if os.path.isfile(p): z.write(p,arcname=os.path.basename(p)); print(f'  + {os.path.basename(p)}')
    if os.path.isdir(CROPPED_PLATES_DIR):
        for root,_,files in os.walk(CROPPED_PLATES_DIR):
            for fn in files:
                fp=os.path.join(root,fn); z.write(fp,arcname=os.path.relpath(fp,'/kaggle/working'))
        print(f'  + cropped_plates/ ({len(glob.glob(os.path.join(CROPPED_PLATES_DIR,"*.jpg")))} images)')
print(f'Results packaged: {ZIP_PATH}')
print('Download from the Kaggle Output panel on the right side of the page.')